# Credit ETF research: signal measurements on the clean universe

Read-only measurements for the FINM 35700 project. Every signal is indexed at the decision date T (EOD data
through T-1 plus the day-T 3PM snapshot) and scored against returns from the close of T onward.

- **Universe:** ETFs with complete corporate-action records (`docs/data_gap.md`).
- **Hedged return:** total return minus a duration-matched Treasury return interpolated from SHY, IEI, IEF, TLH, TLT.
- **IC:** mean daily cross-sectional rank correlation of the signal with the forward return. The t-stat uses a
  Newey-West variance with horizon-1 lags, which allows for overlapping forward windows.
- **Numbers are before trading costs and before the 3% volume cap.**

In [ ]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import DataLoader
from src.performance import bill_rate_from_market
from src.research import hazard, merton, signals, stats
from src.research.panels import build_panels, hedged_return
from src.research.universe import HEDGE_ANCHORS, IG_MATURITY, RECOVERY, in_bucket

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 1. Data, clean universe and liquidity

In [ ]:
# load through the scaffold's loader; the data-quality report is built from the full sample
market = DataLoader(ROOT / "data").load()
quality = market.quality.summary()
pd.Series({k: quality[k] for k in ("n_symbols", "n_clean", "n_excluded", "gap_months", "unrecorded_splits")}).to_frame("value")

In [ ]:
# panels for the clean ETFs, labelled by ticker; analytics are range-filtered and spreads de-spiked
p = build_panels(market)
dur, gs, ttm, ylds = p.a("duration"), p.a("gspread"), p.a("avgttm"), p.a("yield")
bill = bill_rate_from_market(market)                    # annualized T-bill rate (decimal), SHV yield lagged one day
cap_mm = 0.03 * (p.volume * p.close).median() / 1e6     # 3% of median daily dollar volume, $MM

universe = pd.DataFrame({"bucket": p.bucket, "duration": dur.median(), "avgttm": ttm.median(),
                         "gspread_bp": gs.median(), "cap_3pct_mm": cap_mm})
universe.sort_values(["bucket", "avgttm"])

In [ ]:
# rates-hedged daily return, and the ETF groups used below
xr = hedged_return(p.ret, dur)
credit = in_bucket(p.tickers, "IG", "FRN", "HY", "HY-Global", "Loan")
ust = in_bucket(p.tickers, "UST")
ig, hy = in_bucket(p.tickers, "IG"), in_bucket(p.tickers, "HY")
liquid = [t for t in credit if cap_mm[t] >= 1.0]       # 3% cap of at least $1MM a day
groups = p.bucket
print("credit:", len(credit), "| liquid credit:", len(liquid), "| IG:", len(ig), "| HY (US):", len(hy), "| Treasuries:", len(ust))
print("liquid:", " ".join(liquid))

In [ ]:
results = []

def ic_rows(label, signal, target, cols, groups=None):
    """Mean rank IC and overlap-adjusted t-stat at 1, 5 and 20 days for the given ETFs."""
    cols = [c for c in cols if c in signal.columns and c in target.columns]
    t = stats.ic_table(signal[cols], target[cols], (1, 5, 20), None if groups is None else groups[cols])
    out = t.reset_index()
    out.insert(0, "n_etfs", len(cols))
    out.insert(0, "signal", label)
    results.append(out)
    return out.set_index(["signal", "n_etfs", "horizon"])

## 2. Long-only reference (not a baseline)

Buy-and-hold of single clean funds, scored net of financing at the T-bill rate. Recorded for comparison only.

In [ ]:
ref = ["SHV", "SHY", "IEF", "TLT", "FLOT", "VCSH", "LQD", "SJNK", "HYG", "BKLN"]
excess = p.ret[ref].sub(bill / 252, axis=0)
cum = p.ret[ref].fillna(0).cumsum()
pd.DataFrame({
    "sharpe_net_of_financing": {t: stats.financing_adjusted_sharpe(p.ret[t], 1.0, bill) for t in ref},
    "excess_return_pct_yr": excess.mean() * 252 * 100,
    "vol_pct_yr": p.ret[ref].std() * np.sqrt(252) * 100,
    "max_drawdown_pct": (cum.cummax() - cum).max() * 100,
    "cap_3pct_mm": cap_mm[ref],
})

## 3. NAV basis (lecture 2)

Basis = close / NAV - 1. Two signals: the discount at the T-1 close, and the discount implied by the day-T 3PM
mid after removing the rates move since the T-1 close. Both are z-scores against the fund's own trailing 60 days.

In [ ]:
basis = signals.nav_basis_bp(p.close, p.nav)
basis_stats = pd.DataFrame({"bucket": p.bucket, "mean_bp": basis.mean(), "sd_bp": basis.std(),
                            "half_life_d": basis.apply(stats.half_life), "cap_3pct_mm": cap_mm})
basis_stats.groupby("bucket").median(numeric_only=True)

In [ ]:
sig_nav = signals.nav_discount_lagged(p)
sig_nav3 = signals.nav_discount_3pm(p)
pd.concat([
    ic_rows("NAV discount at T-1 close", sig_nav, xr, credit, groups),
    ic_rows("NAV discount at T-1 close (liquid)", sig_nav, xr, liquid, groups),
    ic_rows("NAV discount at 3PM", sig_nav3, xr, credit, groups),
    ic_rows("NAV discount at 3PM (liquid)", sig_nav3, xr, liquid, groups),
    ic_rows("NAV discount at 3PM (Treasuries)", sig_nav3, xr, ust),
])

## 4. 3PM factor residual

Residual = move from the T-1 close to 3PM on T, minus its rates part, minus beta x the reference credit ETF's
hedged move (HYG for high yield and loans, LQD for IG and floaters). Beta is a rolling 120-day estimate through
T-1. The signal is minus the residual. Target: the same residual computed on the following close-to-close returns.

In [ ]:
beta = signals.reference_beta(p)
sig_res = signals.residual_3pm_reversal(p)
resid_cc = signals.factor_residual(p, p.ret, beta)             # close-to-close residual
resid_3c = signals.factor_residual(p, p.ret_3pm_to_close, beta)  # 3PM-to-close residual (before the fill)
res_cols = list(sig_res.columns)
print("ETFs:", len(res_cols), "| residual sd by bucket, bp:",
      (signals.factor_residual(p, p.ret_to_3pm_mid, beta).std() * 1e4).groupby(groups[res_cols]).median().round(1).to_dict())
print("IC with the 3PM-to-close residual on the same day (not capturable, the fill is at the close):",
      round(stats.rank_ic(sig_res, resid_3c, groups[res_cols]).mean(), 3))
pd.concat([
    ic_rows("3PM residual reversal", sig_res, resid_cc, res_cols, groups),
    ic_rows("3PM residual reversal (liquid)", sig_res, resid_cc, [c for c in res_cols if c in liquid], groups),
])

## 5. Hazard-curve edges (lectures 4-5)

Each day a Nelson-Siegel curve is fitted to g-spread against average maturity across the funds of a tier
(weights: sqrt of trailing dollar volume / duration). Edge = market spread - fitted spread, in bp. With the
credit triangle, the fitted curve divided by (1 - R) is the tier's average-hazard curve.

In [ ]:
# fit quality for a grid of the curve-location parameter (chosen on the full sample)
grid = pd.DataFrame({
    "IG, 3 terms": {lam: signals.hazard_edge_fit(p, ig, lam, 3).rmse.median() for lam in (1, 2, 3, 5, 8)},
    "HY, 2 terms": {lam: signals.hazard_edge_fit(p, hy, lam, 2).rmse.median() for lam in (1, 2, 3, 5, 8)},
})
grid.index.name = "lambda"
grid.round(2)

In [ ]:
fit_ig = signals.hazard_edge_fit(p, ig, lam=float(grid["IG, 3 terms"].idxmin()), n_terms=3)
fit_hy = signals.hazard_edge_fit(p, hy, lam=float(grid["HY, 2 terms"].idxmin()), n_terms=2)

def edge_table(fit, cols):
    dev = fit.edges - fit.edges.rolling(120, min_periods=40).mean().shift(1)
    return pd.DataFrame({"avgttm": ttm[cols].median(), "mean_edge_bp": fit.edges.mean(), "sd_edge_bp": fit.edges.std(),
                         "sd_deviation_bp": dev.std(), "half_life_edge_d": fit.edges.apply(stats.half_life),
                         "half_life_deviation_d": dev.apply(stats.half_life), "cap_3pct_mm": cap_mm[cols]}).sort_values("avgttm")

print("IG: lambda", fit_ig.lam, "| median daily RMSE", round(fit_ig.rmse.median(), 1), "bp | days fitted", int(fit_ig.rmse.notna().sum()))
edge_table(fit_ig, ig)

In [ ]:
print("HY: lambda", fit_hy.lam, "| median daily RMSE", round(fit_hy.rmse.median(), 1), "bp | days fitted", int(fit_hy.rmse.notna().sum()))
edge_table(fit_hy, hy)

In [ ]:
# signals known at T: the edge's deviation from its own trailing mean (z-score), and the raw edge
z_ig, z_hy = signals.edge_deviation_z(fit_ig.edges), signals.edge_deviation_z(fit_hy.edges)
raw_ig, raw_hy = fit_ig.edges.shift(1), fit_hy.edges.shift(1)
pd.concat([
    ic_rows("Hazard edge deviation z, IG", z_ig, xr, ig),
    ic_rows("Hazard edge deviation z, HY", z_hy, xr, hy),
    ic_rows("Hazard raw edge, IG", raw_ig, xr, ig),
    ic_rows("Hazard raw edge, HY", raw_hy, xr, hy),
])

**Carry or convergence?** A fund that sits wide of the curve earns the extra spread whether or not the edge
closes. Two targets separate the two: (a) the hedged return minus the accrual of the edge itself
(edge / 252 per day), and (b) minus the change in the edge over the holding window.

In [ ]:
def carry_vs_convergence(label, fit, cols):
    lag = fit.edges.shift(1)
    ex_carry = xr[cols] - lag / 1e4 / 252                 # return after removing the accrual of the edge
    rows = [ic_rows(f"{label}: raw edge vs return ex edge accrual", lag, ex_carry, cols)]
    conv = []
    for h in (1, 5, 20):
        closing = -(fit.edges.shift(-h) - fit.edges)       # fall in the edge from close T to close T+h
        ic = stats.rank_ic(lag, closing)
        conv.append({"horizon": h, "mean_ic": ic.mean(), "t_stat": stats.newey_west_tstat(ic, h - 1), "n_days": len(ic)})
    conv = pd.DataFrame(conv)
    conv.insert(0, "n_etfs", len(cols)); conv.insert(0, "signal", f"{label}: raw edge vs fall in the edge")
    results.append(conv)
    return pd.concat([rows[0], conv.set_index(["signal", "n_etfs", "horizon"])])

pd.concat([carry_vs_convergence("IG", fit_ig, ig), carry_vs_convergence("HY", fit_hy, hy)])

## 6. Seniority (lecture 4)

Implied hazard h = g-spread / (1 - R), with R = 0.80 for loans and 0.40 for high-yield bonds. The clean set has
no preferred-stock ETF. Gap = log(h_leg / h_HYG); its z-score (trailing 120 days, known at T) is related to the
forward return of the leg minus HYG, both rates-hedged.

In [ ]:
legs = ["BKLN", "SRLN", "JNK", "USHY", "SPHY", "SHYG", "SJNK"]
haz = pd.DataFrame({t: hazard.implied_hazard(gs[t], RECOVERY[groups[t]]) for t in ["BKLN", "SRLN", "HYG", "JNK", "USHY"]})
haz.groupby(haz.index.year).median().round(0)

In [ ]:
rows = {}
for leg in legs:
    gap = hazard.log_hazard_gap(gs[leg], gs["HYG"], RECOVERY[groups[leg]], RECOVERY["HY"])
    z = stats.trailing_z(gap, 120, 40).shift(1)
    rel = xr[leg] - xr["HYG"]
    row = {"mean_log_gap": gap.mean(), "sd_log_gap": gap.std(), "half_life_d": stats.half_life(gap),
           "equalizing_recovery": hazard.equalizing_recovery(gs[leg], hazard.implied_hazard(gs["HYG"], RECOVERY["HY"])).median()}
    for h in (1, 5, 20):
        r = stats.predict_ts(z, rel, h)
        row[f"corr_{h}d"], row[f"t_{h}d"] = r["corr"], r["t_stat"]
    for h in (1, 5):                                     # same with the z-score clipped at +/-3
        r = stats.predict_ts(z.clip(-3, 3), rel, h)
        row[f"clipped_corr_{h}d"], row[f"clipped_t_{h}d"] = r["corr"], r["t_stat"]
    rows[f"{leg} vs HYG"] = row
seniority = pd.DataFrame(rows).T
seniority

## 7. Roll-down (lecture 1)

Expected return = carry + roll-down, where roll-down = duration x slope of a fitted curve at the fund's
maturity. Signals and targets are per year of duration, so the comparison is DV01-neutral.

In [ ]:
# rates curve: Nelson-Siegel fit of yield against average maturity across the clean Treasury ETFs
fit_ust = hazard.fit_curve_panel(ylds[ust], ttm[ust], None, lam=2.0, n_terms=3)
roll_ust = signals.rolldown_pct(fit_ust, ttm[ust], dur[ust])                 # % of price per year
curve_ust = [t for t in ust if dur[t].median() >= 1.0]                        # leave out the T-bill fund
carry_ust = ylds[ust].sub(bill * 100, axis=0)                                 # yield over the bill rate, %
sig_roll_ust = ((carry_ust + roll_ust) / dur[ust]).shift(1)
tgt_ust = p.ret[ust].sub(bill / 252, axis=0) / dur[ust].shift(1)              # excess return per year of duration
print("curve fit RMSE, bp:", round(fit_ust.rmse.median() * 100, 1), "| ETFs ranked:", " ".join(curve_ust))
pd.DataFrame({"avgttm": ttm[ust].median(), "duration": dur[ust].median(), "yield_pct": ylds[ust].median(),
              "rolldown_pct_yr": roll_ust.median(), "rolldown_by_year_2023": roll_ust[roll_ust.index.year == 2023].median(),
              "rolldown_by_year_2026": roll_ust[roll_ust.index.year == 2026].median()}).sort_values("avgttm")

In [ ]:
# IG spread curve: the section-5 fit of g-spread against maturity gives the spread slope
roll_ig = signals.rolldown_pct(fit_ig, ttm[ig], dur[ig], per_bp=True)        # % of price per year
sig_roll_ig = ((gs[ig] / 100 + roll_ig) / dur[ig]).shift(1)                   # spread carry + roll-down per year of duration
tgt_ig = xr[ig] / dur[ig].shift(1)
pd.concat([
    ic_rows("Rates carry + roll-down per unit duration", sig_roll_ust, tgt_ust, curve_ust),
    ic_rows("Rates roll-down only per unit duration", (roll_ust / dur[ust]).shift(1), tgt_ust, curve_ust),
    ic_rows("IG spread carry + roll-down per unit duration", sig_roll_ig, tgt_ig, ig),
    ic_rows("IG spread roll-down only per unit duration", (roll_ig / dur[ig]).shift(1), tgt_ig, ig),
])

## 8. Merton model on HYG (lecture 6)

Observables each day: the g-spread, and the credit-only volatility of the debt claim (3M option-implied vol with
the duration-matched Treasury-implied rates vol removed, using a rolling 250-day rate-spread correlation, -0.2
before one is available). Unknowns: leverage L = K/A and asset vol. T = average maturity, r = yield - g-spread.

In [ ]:
anch = list(HEDGE_ANCHORS)

def credit_vol(t, tenor="vol3M"):
    """Credit-only price vol of ETF t (decimal), dated at the close it is observed on."""
    rates_only = merton.rates_vol(dur[t], p.vol[tenor][anch], dur[anch])
    tsy_yield = ylds[t] - gs[t] / 100
    rho = tsy_yield.diff().rolling(250, min_periods=60).corr(gs[t].diff()).shift(1).fillna(-0.2)
    return merton.credit_only_vol(p.vol[tenor][t], rates_only, rho) / 100

T_hyg, r_hyg = ttm["HYG"], (ylds["HYG"] - gs["HYG"] / 100) / 100
cv_hyg = credit_vol("HYG")
sol = merton.solve_panel(gs["HYG"] / 1e4, cv_hyg, T_hyg, r_hyg)
print("credit vol defined on", round(cv_hyg.notna().mean() * 100, 1), "% of days; solved on", round(sol.L.notna().mean() * 100, 1), "%")
pd.concat([sol, (cv_hyg * 100).rename("credit_vol_pct"), gs["HYG"].rename("gspread_bp"), T_hyg.rename("T")], axis=1).groupby(sol.index.year).median()

In [ ]:
# (a) state variables as signals on HYG against Treasuries; plain spread and vol z-scores for comparison
state = {"leverage z": sol.L, "asset vol z": sol.sigma_a, "spread z": gs["HYG"], "credit vol z": cv_hyg}
rows = {}
for name, s in state.items():
    z = stats.trailing_z(s, 120, 40).shift(1)
    row = {}
    for h in (1, 5, 20):
        r = stats.predict_ts(z, xr["HYG"], h)
        row[f"corr_{h}d"], row[f"t_{h}d"], row["n"] = r["corr"], r["t_stat"], r["n"]
    rows[name] = row
pd.DataFrame(rows).T

**Robustness of the state-variable correlations.** These are Pearson correlations of z-scores, so a few extreme
days can drive them. The table repeats them with the z-score clipped at +/-3, with the ten most influential days
removed, and without 2020; the second table is the next-day correlation by year.

In [ ]:
f1 = xr["HYG"].shift(-1)
rows, by_year = {}, {}
for name, s in state.items():
    z = stats.trailing_z(s, 120, 40).shift(1)
    d = pd.concat([z, f1], axis=1, keys=["z", "f"]).dropna()
    contrib = (d.z - d.z.mean()) * (d.f - d.f.mean())
    top = contrib.abs().nlargest(10).index
    clip1, clip5 = stats.predict_ts(z.clip(-3, 3), xr["HYG"], 1), stats.predict_ts(z.clip(-3, 3), xr["HYG"], 5)
    rows[name] = {"max_abs_z": z.abs().max(), "corr_1d": d.corr().iloc[0, 1],
                  "clipped_corr_1d": clip1["corr"], "clipped_t_1d": clip1["t_stat"],
                  "clipped_corr_5d": clip5["corr"], "clipped_t_5d": clip5["t_stat"],
                  "share_from_top10_days": contrib[top].sum() / contrib.sum(),
                  "corr_1d_ex_top10": d.drop(top).corr().iloc[0, 1], "corr_1d_ex_2020": d[d.index.year > 2020].corr().iloc[0, 1]}
    by_year[name] = d.groupby(d.index.year).apply(lambda g: g.z.corr(g.f))
display(pd.DataFrame(rows).T)
pd.DataFrame(by_year)

In [ ]:
# does the day's (spread, credit vol) pair pin down a single (L, asset vol)? solve from six starting points
starts = [(-1.0, -1.0), (0.0, -2.0), (0.3, -3.0), (0.6, -3.5), (-0.5, -0.5), (1.0, -2.5)]
sample = sol.dropna().index[::40]
n_multi = 0
for day in sample:
    roots = {tuple(np.round(merton.solve_leverage_vol(gs["HYG"][day] / 1e4, cv_hyg[day], T_hyg[day], r_hyg[day], x0), 3)) for x0 in starts}
    n_multi += len({x for x in roots if np.isfinite(x[0])}) > 1
print("dates checked:", len(sample), "| dates with more than one solution:", n_multi)

In [ ]:
# (b) tier model in leverage space: HYG's asset vol (20-day median) applied to each fund; edge = the fund's
#     implied log leverage against its own trailing 60 days. Benchmark: the spread gap to HYG, same z-score.
tier = ["JNK", "USHY", "SPHY", "SHYG", "SJNK", "BKLN", "SRLN"]
sig_a = sol.sigma_a.rolling(20, min_periods=5).median()
log_lev = pd.DataFrame({t: np.log(merton.implied_leverage_panel(gs[t] / 1e4, sig_a, ttm[t], (ylds[t] - gs[t] / 100) / 100)) for t in tier})
edge_lev = stats.trailing_z(log_lev, 60, 20).shift(1)
edge_naive = stats.trailing_z(gs[tier].sub(gs["HYG"], axis=0), 60, 20).shift(1)
rel = xr[tier].sub(xr["HYG"], axis=0)
print("median implied leverage:", log_lev.apply(np.exp).median().round(3).to_dict())
print("sd of the leverage edge (z):", round(edge_lev.stack().std(), 2), "| correlation with the naive edge:", round(edge_lev.stack().corr(edge_naive.stack()), 3))
pd.concat([
    ic_rows("Merton leverage edge vs HYG", edge_lev, rel, tier),
    ic_rows("Naive spread-gap edge vs HYG", edge_naive, rel, tier),
])

In [ ]:
# (c) vol term structure: model option vol on the debt claim at 6M and 12M against the market's credit-only vol
rows = {}
for tenor, tau in (("vol6M", 0.5), ("vol12M", 1.0)):
    mkt = credit_vol("HYG", tenor) * 100
    mdl = pd.Series([merton.debt_option_implied_vol(L, s, T, r, tau) for L, s, T, r in zip(sol.L, sol.sigma_a, T_hyg, r_hyg)], index=sol.index) * 100
    edge = mkt - mdl
    z = stats.trailing_z(edge, 120, 40).shift(1)
    row = {"market_median": mkt.median(), "model_median": mdl.median(), "edge_sd": edge.std(), "half_life_d": stats.half_life(edge)}
    for h in (1, 5, 20):
        r = stats.predict_ts(z, xr["HYG"], h)
        row[f"corr_{h}d"], row[f"t_{h}d"] = r["corr"], r["t_stat"]
    for h in (1, 5):                                     # same with the z-score clipped at +/-3
        r = stats.predict_ts(z.clip(-3, 3), xr["HYG"], h)
        row[f"clipped_corr_{h}d"], row[f"clipped_t_{h}d"] = r["corr"], r["t_stat"]
    rows[tenor] = row
pd.DataFrame(rows).T

## 9. All cross-sectional results

In [ ]:
summary = pd.concat(results, ignore_index=True)
summary["ic (t)"] = summary.apply(lambda r: f"{r.mean_ic:+.3f} ({r.t_stat:+.1f})", axis=1)
summary.pivot_table(index=["signal", "n_etfs"], columns="horizon", values="ic (t)", aggfunc="first", sort=False)